# Session 4 Research Report
## Edge Stacking, the Swap Gate, and the Project Verdict

**Date:** 2026-06-05  
**Goal this session:** find an edge that passes an FTMO challenge in 1-2 months with >50% probability.  
**Verdict:** Reached a *gross* combined Sharpe of 0.97 by stacking uncorrelated edges — but **real swap/financing costs cut it to ~0.5 net, below the ~1.0-1.5 required.** A >50% / 2-month FTMO pass is **not achievable** with classical/documented edges on free retail data, net of real costs. Next: go beyond free retail data.

---

## Table of Contents
1. [The FTMO speed spec — required Sharpe](#1)
2. [Edge stacking — the diversification that worked](#2)
3. [Split-half validation of the stack](#3)
4. [The swap gate — the math of holding costs](#4)
5. [Real broker swap rates from MT5](#5)
6. [Net verdict and the structural gap](#6)
7. [Deliverable: the MT5 EA, and the next direction](#7)

In [1]:
import sys
from pathlib import Path
ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'scripts'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CACHE = ROOT / 'data_cache'
print('Setup OK')

Setup OK


<a id='1'></a>
## 1. The FTMO Speed Spec — Required Sharpe

Passing in 1 month vs 1 year are different problems. To make **+10%** in ~30-40 days you need high *drift*, which forces high leverage, which collides with the **−5% daily** and **−10% total** limits. For each Sharpe there is an *optimal* leverage (too little = too slow; too much = breach) and a corresponding *maximum* pass probability.

We Monte-Carlo'd it: simulate daily returns $r_t \sim N(\mu, \sigma)$ with $\mu = S\,\sigma/\sqrt{252}$, barriers at +10% / −10% / −5%-daily, and search leverage for the max pass rate.

In [2]:
rng = np.random.default_rng(0)
def p_pass(ann_sharpe, days, n=8000):
    best = 0
    for vol in np.arange(0.015, 0.028, 0.002):
        mu = ann_sharpe*vol/np.sqrt(252); P=0
        for _ in range(n):
            eq=0.0; ok=False
            for d in range(days):
                x=rng.normal(mu,vol)
                if x<=-0.05: break
                eq+=x
                if eq<=-0.10: break
                if eq>=0.10 and d>=3: ok=True; break
            P+=ok
        best=max(best,P/n)
    return best
print('Max P(pass) at optimal leverage:\n')
print(f"{'Sharpe':>7} {'30d (1mo)':>10} {'40d (2mo)':>10} {'60d (3mo)':>10}")
for S in [0.5,0.7,1.0,1.5,2.0]:
    print(f'{S:>7.1f} {p_pass(S,30)*100:>9.0f}% {p_pass(S,40)*100:>9.0f}% {p_pass(S,60)*100:>9.0f}%')
print('\n=> >50% pass needs Sharpe ~1.5 (1mo), ~1.0 (2mo), ~0.7 (3mo).')

Max P(pass) at optimal leverage:

 Sharpe  30d (1mo)  40d (2mo)  60d (3mo)
    0.5        40%        44%        48%
    0.7        42%        46%        51%
    1.0        45%        50%        56%
    1.5        51%        56%        64%
    2.0        57%        62%        71%

=> >50% pass needs Sharpe ~1.5 (1mo), ~1.0 (2mo), ~0.7 (3mo).


**The spec:** to pass in 1-2 months with >50% probability we need a net Sharpe of roughly **1.0 (2 months) to 1.5 (1 month)**. Our best combined book at the start of the session was 0.71 — close, but short. The plan: stack more uncorrelated edges.

<a id='2'></a>
## 2. Edge Stacking — the Diversification That Worked

Combining $N$ uncorrelated edges, each with Sharpe $s$ and average pairwise correlation $\bar\rho$:

$$S_{\text{combined}} = s \cdot \sqrt{\frac{N}{1 + (N-1)\bar\rho}}$$

With $\bar\rho \approx 0.1$, going from $N=2$ to $N=4$ lifts the Sharpe by ~$\sqrt{2}$. We added two edges chosen to be uncorrelated *by construction*:
- **Crypto trend** (long BTC above its 50-day MA) — crypto *trends* where FX *reverts*.
- **Turn-of-month** seasonality (long an index basket on the last + first 3 trading days).

In [3]:
from quantresearch.cross_sectional import build_currency_returns
from quantresearch.edges import daily_reversion_pnl, is_fx, basket, crypto_trend, turn_of_month

data_fx = {f.stem[:-3]: pd.read_parquet(f) for f in CACHE.glob('*_H1.parquet') if is_fx(f.stem[:-3])}
curr = build_currency_returns(data_fx)
rev = daily_reversion_pnl(curr,12,2,1.5); rev.index = pd.to_datetime(rev.index).normalize()
ovn = basket(use_filter=False); ovn.index = pd.to_datetime(ovn.index).normalize()
edges = {'FX_reversion':rev, 'index_overnight':ovn, 'crypto_trend':crypto_trend(), 'turn_of_month':turn_of_month()}
def sh(r): return r.mean()/r.std()*np.sqrt(252) if r.std() else 0
D = pd.DataFrame(edges).dropna()
stack = sum(D[c]/D[c].std() for c in D)/4
print('GROSS (spread only) edge Sharpes and correlation to the 2-edge book:')
book2 = (D['FX_reversion']/D['FX_reversion'].std()+D['index_overnight']/D['index_overnight'].std())/2
for c in D:
    print(f'  {c:<16} Sharpe {sh(D[c]):5.2f}  corr_book {D[c].corr(book2):+.2f}')
print(f'\n  4-EDGE STACK gross Sharpe: {sh(stack):.2f}  (was 0.71 with 2 edges)')

GROSS (spread only) edge Sharpes and correlation to the 2-edge book:
  FX_reversion     Sharpe  0.45  corr_book +0.77
  index_overnight  Sharpe  0.63  corr_book +0.77
  crypto_trend     Sharpe  0.85  corr_book +0.03
  turn_of_month    Sharpe  0.27  corr_book +0.22

  4-EDGE STACK gross Sharpe: 0.97  (was 0.71 with 2 edges)


C:\Users\noaro\QUANT\Quant-Research\FTMO_Edge_Research\quantresearch\edges.py:118: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return ret, pos.reindex(ret.index).fillna(False).astype(bool)


<a id='3'></a>
## 3. Split-Half Validation of the Stack

A high Sharpe is meaningless if it lives in one period. Every edge — and the stack — must be positive in *both* halves of history.

In [4]:
def split(r):
    r=r.dropna(); m=r.index[len(r)//2]
    return sh(r[r.index<m]), sh(r[r.index>=m])
print(f"{'edge':<16} {'full':>6} {'H1':>6} {'H2':>6}")
for c in D:
    h1,h2=split(D[c]); print(f'  {c:<14} {sh(D[c]):>6.2f} {h1:>6.2f} {h2:>6.2f}')
h1,h2=split(stack)
print(f'  {"STACK":<14} {sh(stack):>6.2f} {h1:>6.2f} {h2:>6.2f}')
print('\nAll positive in both halves; crypto holds in H2 (2022 crash era) -> not a 2020-21 bull artifact.')

edge               full     H1     H2
  FX_reversion     0.45   0.43   0.48
  index_overnight   0.63   0.76   0.46
  crypto_trend     0.85   0.92   0.78
  turn_of_month    0.27   0.21   0.34
  STACK            0.97   1.03   0.90

All positive in both halves; crypto holds in H2 (2022 crash era) -> not a 2020-21 bull artifact.


<a id='4'></a>
## 4. The Swap Gate — the Math of Holding Costs

Our backtests charged **spread** but not **swap/financing** — the daily cost of *holding* a CFD overnight. For multi-day and overnight strategies this can be the *dominant* cost.

Swap is a per-day charge on the notional held:
$$\text{daily cost (bps)} \approx \frac{\text{interest rate} + \text{broker markup} - \text{dividend yield}}{360}\times 10^4$$

The decisive quantity is **return-per-held-day vs cost-per-held-day**. An edge whose daily return barely exceeds its financing dies; one whose return dwarfs it survives. We applied realistic swap on each edge's actual hold-days:

| edge | hold pattern | swap (bps/day) | why |
|---|---|---|---|
| FX reversion | continuous | −1.0 | longs low-yield, shorts high-yield = negative carry |
| index overnight | every night | −1.8 to −2.1 | financing across the rollover, minus dividends |
| crypto trend | when long | −5 to −8.3 | crypto CFD financing is steep |
| turn-of-month | ~4 days/mo | −1.8 | index financing on hold days |

In [5]:
from quantresearch.edges import crypto_with_mask, tom_with_mask
cry, cry_h = crypto_with_mask(); tom, tom_h = tom_with_mask()
rows=[]
for mult,label in [(0,'gross'),(1,'real swap')]:
    # base-case bps/day; the 'real swap' column uses the MT5-measured rates (sec.5)
    S = {'rev':1.0,'ovn':2.1,'cry':8.33,'tom':2.1}
    rev_n=rev-mult*S['rev']/1e4; ovn_n=ovn-mult*S['ovn']/1e4
    cry_n=cry-mult*S['cry']/1e4*cry_h.astype(float); tom_n=tom-mult*S['tom']/1e4*tom_h.astype(float)
    Dn=pd.DataFrame({'rev':rev_n,'ovn':ovn_n,'cry':cry_n,'tom':tom_n}).dropna()
    stk=sum(Dn[c]/Dn[c].std() for c in Dn)/4
    rows.append({'case':label,'rev':round(sh(rev_n),2),'ovn':round(sh(ovn_n),2),
                 'cry':round(sh(cry_n),2),'tom':round(sh(tom_n),2),'STACK':round(sh(stk),2),
                 'crypto_alone':round(sh(cry_n),2)})
print(pd.DataFrame(rows).to_string(index=False))
print('\nSwap turns the thin continuous-hold edges near-zero; only crypto trend survives.')

     case  rev  ovn  cry  tom  STACK  crypto_alone
    gross 0.46 0.63 0.79 0.30   0.97          0.79
real swap 0.17 0.20 0.49 0.18   0.46          0.49

Swap turns the thin continuous-hold edges near-zero; only crypto trend survives.


C:\Users\noaro\QUANT\Quant-Research\FTMO_Edge_Research\quantresearch\edges.py:118: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return ret, pos.reindex(ret.index).fillna(False).astype(bool)


<a id='5'></a>
## 5. Real Broker Swap Rates from MT5

We didn't have to estimate — MT5 exposes the actual `SWAP_LONG`/`SWAP_SHORT` per symbol (`scripts/pull_swap_rates.py`). The measured rates were *worse* than estimated:
- **BTCUSD: −8.33 bps/day** (−30%/yr), both directions.
- **US100 overnight: −2.1 bps/day** long.
- **FX carry pairs: only +0.1 to +0.28 bps/day** long — the broker skims the short side (−1 to −2 bps), so carry is far thinner than the textbook rate differential.
- **USOIL: +4 bps/day** long (backwardation) — a rare positive-carry hold.

In [6]:
f = CACHE/'swap_rates.parquet'
if f.exists():
    sr = pd.read_parquet(f)
    print('Top positive-carry (earn to hold LONG):')
    print(sr.nlargest(5,'carry_long_bps')[['symbol','carry_long_bps','carry_short_bps']].to_string(index=False))
    print('\nMost expensive to hold:')
    print(sr.nsmallest(5,'carry_long_bps')[['symbol','carry_long_bps','carry_short_bps']].to_string(index=False))
else:
    print('Run scripts/pull_swap_rates.py (needs MT5) to populate swap_rates.parquet')

Top positive-carry (earn to hold LONG):
    symbol  carry_long_bps  carry_short_bps
USOIL.cash           3.989          -18.391
    AUDCHF           0.278           -2.013
    AUDJPY           0.179           -1.539
    USDCHF           0.132           -1.660
    GBPJPY           0.129           -1.290

Most expensive to hold:
    symbol  carry_long_bps  carry_short_bps
    BTCUSD          -8.333           -8.333
US100.cash          -2.107           -0.042
    XAUUSD          -1.990           -0.352
    EURAUD          -1.101            0.139
    NZDUSD          -1.075            0.017


<a id='6'></a>
## 6. Net Verdict and the Structural Gap

### The numbers

| stage | combined Sharpe |
|---|---|
| 2-edge book (gross) | 0.71 |
| 4-edge stack (gross) | 0.97 |
| 4-edge stack (net of real swap) | **0.46** |
| crypto trend alone (net) | **0.56** |
| **Required for >50% / 2-month pass** | **~1.0** |

### Why the gap is structural, not a tuning problem
1. **Swap destroys continuous-hold edges.** Reversion (0.46→0.17) and overnight (0.63→0.20) were financing mirages — their per-day return barely exceeded the per-day cost.
2. **The one swap-robust edge is leverage-capped.** Crypto trend survives (0.56), but FTMO caps crypto at **1:1 leverage** on a Swing account, so it can't be run hot enough to sprint to +10%.
3. **Swap-free intraday edges weren't there.** Market Intraday Momentum and opening-range breakout both came up empty in our data.
4. **Carry is broker-skimmed and un-backtestable** with the data we have (no historical swap series).

### The conclusion
**Net of all realistic costs, classical and documented edges on free retail data cap around Sharpe 0.5. A >50% / 1-2-month FTMO pass requires ~1.0-1.5. The gap cannot be closed by combination, risk management, or tuning** — it requires a stronger *source* of alpha. That scarcity is precisely why funded-trader programs are profitable businesses: the bar is genuinely hard to clear.

<a id='7'></a>
## 7. Deliverable: the MT5 EA, and the Next Direction

### `mt5/TrendFollowEA.mq5`
The one edge that survived real costs — **crypto trend** — is single-symbol and directly backtestable in the MT5 Strategy Tester, which applies the **real −8.33 bps/day swap and live spread**. This independently validates the net-of-cost result.

**To backtest:** compile in MetaEditor, run on **BTCUSD, D1**, modelling *"Every tick based on real ticks"* with real spread and swap enabled. Defaults: 50-day SMA, long/flat, 1% risk, ATR×3 stop. Expect a modest, swap-eroded trend equity curve — the point is to *see the real costs bite*, confirming why this can't sprint through an FTMO challenge.

### Next: beyond free retail data
The honest conclusion points the way. Sharpe ~1.5 edges live where free retail OHLC data doesn't reach:
- **Microstructure / order-flow** (tick data, order book imbalance) — intraday, swap-free, higher Sharpe, but needs L2/tick data and fast execution.
- **Options / volatility** (variance risk premium, vol-selling with tail hedges) — documented high-Sharpe premia, needs options data.
- **Alternative data** (positioning/COT, sentiment, on-chain for crypto) — orthogonal signals.
- **Serious ML on richer feature sets** — many weak features combined, rather than one indicator on hourly bars.

These are larger undertakings with genuinely uncertain payoffs — but they are where the required alpha actually lives. The harness, cost model, and FTMO evaluator we built carry straight over to evaluate whatever edges those sources produce.